# FIRMS MODIS and VIIRS active-fire fields
Please ask your own api keys at https://firms.modaps.eosdis.nasa.gov/api/map_key/
This folder contains FIRMS active-fire detections downloaded over the full Mace Head HYSPLIT trajectory for 2003-2025. Each row is a satellite-detected thermal anomaly or active-fire pixel. The point location is the centre of the flagged pixel, not the physical fire perimeter. MODIS active-fire pixels are approximately 1 km at nadir, while VIIRS active-fire pixels are approximately 375 m at nadir. Pixel size increases away from nadir, which is why `scan` and `track` must be retained for interpretation. NASA describes FIRMS detections as active fires or other thermal anomalies, including vegetation fires, volcanoes, static land sources, and offshore sources.

## Columns used for interpretation

| Column | Meaning | Units or coding | Scientific interpretation |
|---|---|---:|---|
| `brightness` | MODIS channel 21/22 brightness temperature or VIIRS I4 brightness temperature, depending on product | K | Thermal intensity of the fire pixel in the mid-infrared. Larger values indicate a stronger sub-pixel thermal anomaly, but values are sensor-dependent and should not be compared blindly between MODIS and VIIRS. |
| `scan` | Across-track pixel size | km | Width of the satellite pixel footprint. Larger `scan` means a larger ground footprint and more spatial uncertainty. Use this when weighting or filtering detections near trajectory paths. |
| `track` | Along-track pixel size | km | Length of the satellite pixel footprint. Together with `scan`, gives approximate pixel area. Pixel area can be approximated as `scan × track`. |
| `acq_date` | Satellite acquisition date | UTC date | Date of satellite overpass. Use with `acq_time` to create a proper UTC timestamp. |
| `acq_time` | Satellite acquisition time | UTC, HHMM integer | Time of overpass. Values such as `40` mean `00:40 UTC`; values such as `502` mean `05:02 UTC`. Pad to four digits before parsing. |
| `confidence` | Detection confidence | MODIS: 0 to 100 percent; VIIRS: `l`, `n`, `h` | Quality indicator for each detected pixel. MODIS confidence is numeric; VIIRS confidence is categorical. |
| `version` | Processing collection and source | Text or number | Identifies product collection and processing stream. Standard processing is preferable for retrospective analysis; near-real-time products are useful operationally but may be revised. |
| `bright_t31` | MODIS channel 31 or VIIRS I5 brightness temperature | K | Longwave infrared background/fire-pixel temperature. Useful with `brightness` to distinguish thermal contrast and contextual anomaly strength. |
| `frp` | Fire radiative power | MW | Pixel-integrated radiative power. This is the most useful continuous proxy for combustion intensity and emissions potential, although it is affected by cloud, view geometry, saturation, and sub-pixel fire structure. |
| `type` | Inferred hotspot type | 0, 1, 2, 3 | For MODIS/FIRMS: `0 = presumed vegetation fire`, `1 = active volcano`, `2 = other static land source`, `3 = offshore`. For biomass-burning aerosol interpretation, usually retain `type == 0`; treat `type == 2` and `type == 3` separately or remove them. |

## Confidence filtering

### MODIS

MODIS `confidence` ranges from 0 to 100 percent. The MODIS fire product user guide defines the classes as:

| MODIS confidence | Class | Suggested use |
|---:|---|---|
| `0 ≤ confidence < 30` | Low | Highest false-alarm risk; use only in sensitivity tests or when maximum detectability is required. |
| `30 ≤ confidence < 80` | Nominal | Recommended baseline together with high-confidence detections. |
| `80 ≤ confidence ≤ 100` | High | Conservative subset; fewer false positives but more omission of weak or small fires. |

For a defensible baseline, use:

```python
modis = modis[modis["confidence"] >= 30]


Available active-fire sources:
        data_id   min_date   max_date
       MODIS_SP 2000-11-01 2026-02-28
  VIIRS_SNPP_SP 2012-01-20 2026-03-31
VIIRS_NOAA20_SP 2018-04-01 2026-03-31

Date range: 2003-08-01 to 2025-08-31
BBOX: -170.927,23.957,132.153,89.345
Chunk size: up to 5 days
Total tasks: 3161
Pending tasks: 3161
Workers: 12
API keys: 5
Rate limit enforced: 4500 requests per 600 seconds per key

1/3161 MODIS_SP 2003-08-31 to 2003-09-04 done rows=15431 mb=1.65 sec=1.76
2/3161 MODIS_SP 2003-08-06 to 2003-08-10 done rows=12355 mb=1.33 sec=2.21
3/3161 MODIS_SP 2003-08-11 to 2003-08-15 done rows=15632 mb=1.68 sec=2.25
4/3161 MODIS_SP 2003-09-25 to 2003-09-29 done rows=15880 mb=1.69 sec=1.96
5/3161 MODIS_SP 2003-09-10 to 2003-09-14 done rows=10022 mb=1.07 sec=2.97
6/3161 MODIS_SP 2003-09-15 to 2003-09-19 done rows=11888 mb=1.27 sec=3.21
7/3161 MODIS_SP 2003-08-26 to 2003-08-30 done rows=16694 mb=1.78 sec=2.89
8/3161 MODIS_SP 2003-08-01 to 2003-08-05 done rows=15529 mb=1.67 sec=3.03
9/3

In [ ]:
from pathlib import Path
from io import StringIO
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import cycle
from collections import defaultdict, deque
import threading
import time
import random
import requests
import pandas as pd

MAP_KEYS = [
    "f0462fd4fac13d0ecdce07b1453dbaea",
    "aa7e9aa11b1427bccf6d85b3ecb9bc72",
    "7186c287769e330b661daae852f4228b",
    "a09efe882527fbfd7a1c6ee543b26b43",
    "0888bd150471ac9e907fcfffe3a821c5",
]

OUT_DIR = Path(r"D:\FIRMS_fires")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BBOX = "-170.927,23.957,132.153,89.345"
START_DATE = date(2003, 8, 1)
END_DATE = date(2025, 8, 31)
DAY_RANGE_MAX = 5

SCAN_WORKERS = 20
DOWNLOAD_WORKERS = 20

RATE_LIMIT_PER_KEY = 4500
RATE_WINDOW_SECONDS = 600
EXCEEDANCE_SLEEP_SECONDS = 660
REQUEST_TIMEOUT_SECONDS = 420
MAX_HTTP_TRIES = 4
MAX_DEFER_ROUNDS = 200

BASE_AREA = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
BASE_AVAIL = "https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv"

ACTIVE_FIRE_SOURCES = [
    "MODIS_SP",
    "VIIRS_SNPP_SP",
    "VIIRS_NOAA20_SP",
    "VIIRS_NOAA21_SP",
]

REQUIRED_FIRMS_COLUMNS = {
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
}

REQUIRED_ADDED_COLUMNS = {
    "firms_source",
    "request_start_date",
    "request_end_date",
}

key_cycle = cycle(MAP_KEYS)
key_lock = threading.Lock()
rate_lock = threading.Lock()
print_lock = threading.Lock()
file_lock = threading.Lock()
key_windows = defaultdict(deque)

manifest_file = OUT_DIR / "firms_download_manifest.csv"
error_file = OUT_DIR / "firms_download_errors.csv"
deferred_file = OUT_DIR / "firms_download_deferred.csv"
scan_file = OUT_DIR / "firms_existing_file_scan.csv"
corrupt_file = OUT_DIR / "firms_corrupt_files.csv"


class RateExceeded(Exception):
    pass


def next_key():
    with key_lock:
        return next(key_cycle)


def wait_for_key_slot(key):
    while True:
        now = time.time()
        with rate_lock:
            q = key_windows[key]
            while q and now - q[0] > RATE_WINDOW_SECONDS:
                q.popleft()
            if len(q) < RATE_LIMIT_PER_KEY:
                q.append(now)
                return
            sleep_for = RATE_WINDOW_SECONDS - (now - q[0]) + 1
        time.sleep(max(1, sleep_for))


def append_csv(path, row):
    with file_lock:
        df = pd.DataFrame([row])
        header = not path.exists()
        df.to_csv(path, mode="a", header=header, index=False)


def write_csv(path, rows):
    pd.DataFrame(rows).to_csv(path, index=False)


def read_csv_text(txt):
    txt = txt.strip()
    if not txt:
        return pd.DataFrame()
    low = txt[:500].lower()
    if "exceeding allowed transaction limit" in low:
        raise RateExceeded(txt[:1000])
    if low.startswith(("invalid", "error", "exception")):
        raise RuntimeError(txt[:1000])
    return pd.read_csv(StringIO(txt))


def get_text(url, key=None, timeout=REQUEST_TIMEOUT_SECONDS, max_tries=MAX_HTTP_TRIES):
    last_error = None

    for i in range(max_tries):
        if key is not None:
            wait_for_key_slot(key)

        t0 = time.perf_counter()

        try:
            r = requests.get(url, timeout=timeout)
            seconds = time.perf_counter() - t0
            txt = r.text if r.text is not None else ""
            low = txt[:500].lower()

            if "exceeding allowed transaction limit" in low:
                raise RateExceeded(txt[:1000])

            if r.status_code == 200:
                if low.strip().startswith(("invalid", "error", "exception")):
                    raise RuntimeError(txt[:1000])
                return txt, seconds, r.status_code

            if r.status_code == 400 and "exceeding allowed transaction limit" in low:
                raise RateExceeded(txt[:1000])

            if r.status_code in (429, 500, 502, 503, 504):
                last_error = RuntimeError(f"HTTP {r.status_code}: {txt[:1000]}")
                time.sleep((2 ** i) + random.uniform(1, 5))
                continue

            raise RuntimeError(f"HTTP {r.status_code}: {txt[:1000]}")

        except RateExceeded:
            raise

        except Exception as e:
            last_error = e
            time.sleep((2 ** i) + random.uniform(1, 5))

    raise RuntimeError(f"Failed after retries: {last_error}")


def date_chunks_by_year(start, end, step_days):
    y = start.year

    while y <= end.year:
        ys = max(start, date(y, 1, 1))
        ye = min(end, date(y, 12, 31))
        d = ys

        while d <= ye:
            e = min(d + timedelta(days=step_days - 1), ye)
            yield d, e, (e - d).days + 1
            d = e + timedelta(days=1)

        y += 1


def get_availability():
    key = MAP_KEYS[0]
    url = f"{BASE_AVAIL}/{key}/ALL"

    txt, seconds, status = get_text(url, key=key, timeout=180)
    df = read_csv_text(txt)

    df["min_date"] = pd.to_datetime(df["min_date"]).dt.date
    df["max_date"] = pd.to_datetime(df["max_date"]).dt.date
    df = df[df["data_id"].isin(ACTIVE_FIRE_SOURCES)].copy()
    df = df[(df["max_date"] >= START_DATE) & (df["min_date"] <= END_DATE)].copy()

    return df.sort_values(["min_date", "data_id"]).reset_index(drop=True)


def build_tasks(avail):
    tasks = []

    for _, row in avail.iterrows():
        source = row["data_id"]
        s0 = max(START_DATE, row["min_date"])
        s1 = min(END_DATE, row["max_date"])

        for d0, d1, drange in date_chunks_by_year(s0, s1, DAY_RANGE_MAX):
            source_dir = OUT_DIR / str(d0.year) / source
            source_dir.mkdir(parents=True, exist_ok=True)

            out_file = source_dir / f"firms_{source}_{d0:%Y%m%d}_{d1:%Y%m%d}_{drange}d.csv"
            tmp_file = source_dir / f"firms_{source}_{d0:%Y%m%d}_{d1:%Y%m%d}_{drange}d.tmp.csv"

            tasks.append({
                "source": source,
                "start": d0,
                "end": d1,
                "day_range": drange,
                "out_file": out_file,
                "tmp_file": tmp_file,
            })

    return tasks


def unique_corrupt_path(path):
    stem = path.name
    parent = path.parent
    candidate = parent / f"{stem}.corrupt"

    if not candidate.exists():
        return candidate

    i = 1
    while True:
        candidate = parent / f"{stem}.corrupt.{i}"
        if not candidate.exists():
            return candidate
        i += 1


def validate_existing_csv(path):
    if not path.exists():
        return False, "missing"

    size = path.stat().st_size

    if size == 0:
        return False, "zero_bytes"

    try:
        df_head = pd.read_csv(path, nrows=5)
    except Exception as e:
        return False, f"read_error: {repr(e)}"

    cols = set(df_head.columns)

    if not REQUIRED_FIRMS_COLUMNS.issubset(cols):
        missing = sorted(REQUIRED_FIRMS_COLUMNS - cols)
        return False, f"missing_firms_columns: {missing}"

    if not REQUIRED_ADDED_COLUMNS.issubset(cols):
        missing = sorted(REQUIRED_ADDED_COLUMNS - cols)
        return False, f"missing_added_columns: {missing}"

    return True, "valid"


def scan_one_task(task):
    out_file = task["out_file"]
    tmp_file = task["tmp_file"]

    if tmp_file.exists():
        try:
            tmp_file.unlink()
        except Exception:
            pass

    exists = out_file.exists()
    size = out_file.stat().st_size if exists else 0
    valid, reason = validate_existing_csv(out_file)

    moved_to = None

    if exists and not valid:
        bad_file = unique_corrupt_path(out_file)
        try:
            out_file.replace(bad_file)
            moved_to = str(bad_file)
        except Exception as e:
            reason = f"{reason}; move_failed: {repr(e)}"

        append_csv(corrupt_file, {
            "source": task["source"],
            "start_date": task["start"].isoformat(),
            "end_date": task["end"].isoformat(),
            "day_range": task["day_range"],
            "original_file": str(out_file),
            "moved_to": moved_to,
            "bytes": size,
            "reason": reason,
        })

    return {
        "source": task["source"],
        "start_date": task["start"].isoformat(),
        "end_date": task["end"].isoformat(),
        "day_range": task["day_range"],
        "exists": exists,
        "valid": valid,
        "bytes": size,
        "mb": size / 1024 / 1024 if size else 0,
        "file": str(out_file),
        "reason": reason,
        "moved_to": moved_to,
    }


def scan_existing_files(tasks):
    rows = []
    valid_files = set()

    with ThreadPoolExecutor(max_workers=SCAN_WORKERS) as ex:
        futures = {ex.submit(scan_one_task, task): task for task in tasks}
        done = 0

        for fut in as_completed(futures):
            row = fut.result()
            rows.append(row)
            done += 1

            if row["valid"]:
                valid_files.add(row["file"])

            if done % 250 == 0 or done == len(tasks):
                with print_lock:
                    print(f"Scanned {done}/{len(tasks)} expected files")

    write_csv(scan_file, rows)
    return valid_files


def download_task(task):
    source = task["source"]
    d0 = task["start"]
    d1 = task["end"]
    drange = task["day_range"]
    out_file = task["out_file"]
    tmp_file = task["tmp_file"]

    valid, reason = validate_existing_csv(out_file)

    if valid:
        return {
            "source": source,
            "start_date": d0.isoformat(),
            "end_date": d1.isoformat(),
            "day_range": drange,
            "status": "exists",
            "seconds": 0.0,
            "rows": None,
            "mb": out_file.stat().st_size / 1024 / 1024,
            "file": str(out_file),
            "error": None,
        }

    if out_file.exists() and not valid:
        bad_file = unique_corrupt_path(out_file)
        size = out_file.stat().st_size
        try:
            out_file.replace(bad_file)
            append_csv(corrupt_file, {
                "source": source,
                "start_date": d0.isoformat(),
                "end_date": d1.isoformat(),
                "day_range": drange,
                "original_file": str(out_file),
                "moved_to": str(bad_file),
                "bytes": size,
                "reason": reason,
            })
        except Exception:
            pass

    if tmp_file.exists():
        try:
            tmp_file.unlink()
        except Exception:
            pass

    key = next_key()
    url = f"{BASE_AREA}/{key}/{source}/{BBOX}/{drange}/{d0:%Y-%m-%d}"

    t0 = time.perf_counter()
    txt, request_seconds, http_status = get_text(url, key=key)
    total_seconds = time.perf_counter() - t0

    df = read_csv_text(txt)

    if len(df):
        df.insert(0, "firms_source", source)
        df.insert(1, "request_start_date", d0.isoformat())
        df.insert(2, "request_end_date", d1.isoformat())

    df.to_csv(tmp_file, index=False)
    tmp_file.replace(out_file)

    valid_after, reason_after = validate_existing_csv(out_file)

    if not valid_after:
        bad_file = unique_corrupt_path(out_file)
        size = out_file.stat().st_size if out_file.exists() else 0

        if out_file.exists():
            out_file.replace(bad_file)

        append_csv(corrupt_file, {
            "source": source,
            "start_date": d0.isoformat(),
            "end_date": d1.isoformat(),
            "day_range": drange,
            "original_file": str(out_file),
            "moved_to": str(bad_file),
            "bytes": size,
            "reason": f"post_download_validation_failed: {reason_after}",
        })

        raise RuntimeError(f"Downloaded file failed validation: {reason_after}")

    return {
        "source": source,
        "start_date": d0.isoformat(),
        "end_date": d1.isoformat(),
        "day_range": drange,
        "status": "done",
        "seconds": total_seconds,
        "rows": len(df),
        "mb": out_file.stat().st_size / 1024 / 1024,
        "file": str(out_file),
        "error": None,
    }


def run_round(pending, round_id):
    completed = 0
    failed = 0
    deferred = []
    fatal_failed = []

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
        futures = {ex.submit(download_task, task): task for task in pending}

        for fut in as_completed(futures):
            task = futures[fut]

            try:
                row = fut.result()
                completed += 1
                append_csv(manifest_file, row)

                with print_lock:
                    print(
                        f"round={round_id} done={completed} failed={failed} deferred={len(deferred)} "
                        f"{row['source']} {row['start_date']} to {row['end_date']} "
                        f"{row['status']} rows={row['rows']} mb={row['mb']:.2f} sec={row['seconds']:.2f}"
                    )

            except RateExceeded as e:
                failed += 1
                deferred.append(task)

                row = {
                    "source": task["source"],
                    "start_date": task["start"].isoformat(),
                    "end_date": task["end"].isoformat(),
                    "day_range": task["day_range"],
                    "status": "deferred_rate_exceeded",
                    "round": round_id,
                    "seconds": None,
                    "rows": None,
                    "mb": None,
                    "file": str(task["out_file"]),
                    "error": repr(e),
                }

                append_csv(deferred_file, row)

                with print_lock:
                    print(
                        f"round={round_id} deferred={len(deferred)} "
                        f"{task['source']} {task['start']} to {task['end']} rate exceeded"
                    )

            except Exception as e:
                failed += 1
                fatal_failed.append(task)

                row = {
                    "source": task["source"],
                    "start_date": task["start"].isoformat(),
                    "end_date": task["end"].isoformat(),
                    "day_range": task["day_range"],
                    "status": "failed",
                    "round": round_id,
                    "seconds": None,
                    "rows": None,
                    "mb": None,
                    "file": str(task["out_file"]),
                    "error": repr(e),
                }

                append_csv(error_file, row)

                with print_lock:
                    print(
                        f"round={round_id} failed "
                        f"{task['source']} {task['start']} to {task['end']}: {e}"
                    )

    return completed, failed, deferred, fatal_failed


def unfinished_tasks(tasks):
    missing = []

    for t in tasks:
        valid, reason = validate_existing_csv(t["out_file"])
        if not valid:
            missing.append(t)

    return missing


def main():
    t_all = time.perf_counter()

    avail = get_availability()

    print("Available active-fire sources:")
    print(avail[["data_id", "min_date", "max_date"]].to_string(index=False))
    print()

    tasks = build_tasks(avail)

    print(f"Date range: {START_DATE} to {END_DATE}")
    print(f"BBOX: {BBOX}")
    print(f"Chunk size: up to {DAY_RANGE_MAX} days")
    print(f"Total expected files: {len(tasks)}")
    print(f"Scan workers: {SCAN_WORKERS}")
    print(f"Download workers: {DOWNLOAD_WORKERS}")
    print(f"API keys: {len(MAP_KEYS)}")
    print(f"Per-key rate guard: {RATE_LIMIT_PER_KEY} requests per {RATE_WINDOW_SECONDS} seconds")
    print()

    valid_files = scan_existing_files(tasks)

    pending = [
        t for t in tasks
        if str(t["out_file"]) not in valid_files
    ]

    print()
    print(f"Already downloaded valid files: {len(valid_files)}")
    print(f"Pending downloads after scan: {len(pending)}")
    print(f"Scan CSV: {scan_file}")
    print()

    round_id = 1
    total_completed = 0
    total_fatal_failed = 0
    total_deferred_events = 0

    while pending and round_id <= MAX_DEFER_ROUNDS:
        print(f"Starting download round {round_id} with {len(pending)} pending tasks")

        completed, failed, deferred, fatal_failed = run_round(pending, round_id)

        total_completed += completed
        total_fatal_failed += len(fatal_failed)
        total_deferred_events += len(deferred)

        pending = unfinished_tasks(tasks)

        if pending and deferred:
            print()
            print(f"Round {round_id} hit FIRMS transaction exceedance.")
            print(f"Remaining unfinished files: {len(pending)}")
            print(f"Sleeping {EXCEEDANCE_SLEEP_SECONDS} seconds before retrying unfinished chunks.")
            time.sleep(EXCEEDANCE_SLEEP_SECONDS)

        elif pending and not deferred:
            print()
            print(f"Round {round_id} ended with unfinished files but no rate-exceedance signal.")
            print(f"Remaining unfinished files: {len(pending)}")
            print("Stopping to avoid looping on structural errors.")
            break

        round_id += 1

    elapsed = time.perf_counter() - t_all
    pending_final = unfinished_tasks(tasks)

    print()
    print("Done")
    print(f"Completed or already present: {len(tasks) - len(pending_final)}")
    print(f"Still missing: {len(pending_final)}")
    print(f"Fatal failures logged: {total_fatal_failed}")
    print(f"Deferred rate-exceedance events logged: {total_deferred_events}")
    print(f"Elapsed minutes: {elapsed / 60:.2f}")
    print(f"Manifest: {manifest_file}")
    print(f"Deferred: {deferred_file}")
    print(f"Errors: {error_file}")
    print(f"Scan: {scan_file}")
    print(f"Corrupt files: {corrupt_file}")

    if pending_final:
        missing_file = OUT_DIR / "firms_still_missing_after_run.csv"

        rows = [
            {
                "source": t["source"],
                "start_date": t["start"].isoformat(),
                "end_date": t["end"].isoformat(),
                "day_range": t["day_range"],
                "file": str(t["out_file"]),
            }
            for t in pending_final
        ]

        write_csv(missing_file, rows)
        print(f"Still missing CSV: {missing_file}")


if __name__ == "__main__":
    main()

Available active-fire sources:
        data_id   min_date   max_date
       MODIS_SP 2000-11-01 2026-02-28
  VIIRS_SNPP_SP 2012-01-20 2026-03-31
VIIRS_NOAA20_SP 2018-04-01 2026-03-31

Date range: 2003-08-01 to 2025-08-31
BBOX: -170.927,23.957,132.153,89.345
Chunk size: up to 5 days
Total expected files: 3161
Scan workers: 20
Download workers: 20
API keys: 5
Per-key rate guard: 4500 requests per 600 seconds

Scanned 250/3161 expected files
Scanned 500/3161 expected files
Scanned 750/3161 expected files
Scanned 1000/3161 expected files
Scanned 1250/3161 expected files
Scanned 1500/3161 expected files
Scanned 1750/3161 expected files
Scanned 2000/3161 expected files
Scanned 2250/3161 expected files
Scanned 2500/3161 expected files
Scanned 2750/3161 expected files
Scanned 3000/3161 expected files
Scanned 3161/3161 expected files

Already downloaded valid files: 534
Pending downloads after scan: 2627
Scan CSV: D:\FIRMS_fires\firms_existing_file_scan.csv

Starting download round 1 with 2627 